# IKG Column Lineage Explorer

Two modes — use whichever fits your workflow:

| Mode | Description |
|------|-------------|
| **Column / Table** | Pick a column from a searchable dropdown, pick a profile table, click Trace → saves `<table>_<column>_lineage.html` |
| **Insight Type** | Pick an ODM insight type, click Trace → saves `<insight_type>_lineage.html` covering all its rule columns |

**Prerequisites:** `USE_GREENPLUM = True` (both modes query live Greenplum tables)


## 1. Imports

In [ ]:
import pandas as pd
import json
import os
import re
import datetime
from pathlib import Path
from collections import deque
from IPython.display import display as ipy_display, clear_output
import ipywidgets as widgets


## 2. Configuration

In [ ]:
# ── Greenplum connection ─────────────────────────────────────────────
GP_HOST = 'greenplum-rdsp.zur.swissbank.com'
GP_PORT = 5432
GP_DB   = 'gprdsp'
GP_USER = 'ds_rdsp_dev'
GP_PASSWORD = ''  # leave blank to be prompted

# ── Schema / table names ─────────────────────────────────────────────
IKG_SCHEMA = 'sandbox_prj_smart_insights'
ODM_SCHEMA = 'core_ikg'
LIN_TABLE  = 'ikg_column_lineage_master_auto_refresh'
ODM_TABLE  = 'odm_rule_metadata_auto_refresh'


## 3. Connect

In [ ]:
import getpass, sqlalchemy
if not GP_PASSWORD:
    GP_PASSWORD = getpass.getpass('Greenplum password: ')
engine = sqlalchemy.create_engine(
    f'postgresql+psycopg2://{GP_USER}:{GP_PASSWORD}@{GP_HOST}:{GP_PORT}/{GP_DB}'
)
print('Connected to Greenplum')


## 4. Load Data

In [ ]:
# ── Lineage table ────────────────────────────────────────────────────
df_lin = pd.read_sql(
    f'SELECT * FROM {IKG_SCHEMA}.{LIN_TABLE}', engine
).fillna('')
print(f'Lineage table: {len(df_lin):,} rows')

# ── Profile rows (target_table = sub_target_table ending _profile_curr_ikg) ─
def is_profile(r):
    tt  = str(r.get('target_table',     '') or '').lower().strip()
    stt = str(r.get('sub_target_table', '') or '').lower().strip()
    return tt == stt and tt.endswith('_profile_curr_ikg')

_profile_mask = df_lin.apply(is_profile, axis=1)
df_profile = df_lin[_profile_mask]

# ── Column list from profile rows (searchable dropdown) ───────────────
COLS_LIST = sorted([
    c for c in df_profile['target_column'].dropna().unique().tolist()
    if c and str(c).strip()
])
print(f'Profile columns available: {len(COLS_LIST)}')

# ── Insight types from ODM metadata ──────────────────────────────────
try:
    df_it = pd.read_sql(
        f'SELECT DISTINCT insight_type FROM {ODM_SCHEMA}.{ODM_TABLE}'
        ' WHERE insight_type IS NOT NULL ORDER BY insight_type',
        engine
    )
    IT_LIST = [it for it in df_it['insight_type'].tolist() if it and str(it).strip()]
    print(f'Insight types available: {len(IT_LIST)}')
except Exception as e:
    IT_LIST = []
    print(f'Could not load insight types: {e}')


## 5. Helpers

In [ ]:
# ── Fast lookup helpers (vectorised over pre-built list) ─────────────
_rows = df_lin.to_dict('records')

def _get(tbl, col):
    t, c = tbl.lower(), col.lower()
    return [r for r in _rows
            if str(r.get('sub_target_table','') or '').lower() == t
            and str(r.get('target_column',   '') or '').lower() == c]

def _schema(tbl):
    t = tbl.lower()
    for r in _rows:
        if str(r.get('source_table','') or '').lower() == t and r.get('source_schema'):
            return r['source_schema']
        if str(r.get('sub_target_table','') or '').lower() == t and r.get('sub_target_schema'):
            return r['sub_target_schema']
    return ''

def trace(col, tbl):
    """BFS lineage trace from profile table/column back to all sources."""
    nid_f = lambda a, b: f'{a.lower()}::{b.lower()}'
    nodes, nrows, edges, visited = {}, {}, {}, set()
    q = deque()

    def add(t, c, sc, start=False):
        k = nid_f(t, c)
        if k not in nodes:
            nodes[k] = {'id':k,'tbl':t,'col':c,'schema':sc,'is_start':start}
            nrows[k] = []
        return k

    # Seed from the chosen profile table + column
    seed = [r for r in _rows if is_profile(r)
            and str(r.get('target_table',  '') or '').lower() == tbl.lower()
            and str(r.get('target_column', '') or '').lower() == col.lower()]
    if not seed:
        return None

    sid = add(tbl, col, seed[0].get('target_schema',''), True)
    nrows[sid] = seed
    visited.add(sid)
    for r in seed:
        q.append((r.get('source_table','') or '',
                  r.get('source_column','') or '', sid, [r]))

    itr = 0
    while q and itr < 800:
        itr += 1
        stbl, scol, pid, tr = q.popleft()

        # Blank source_table: find scol as target_column in profile rows
        if not stbl and scol:
            up = [r for r in _rows if is_profile(r)
                  and str(r.get('target_column','') or '').lower() == scol.lower()]
            for r in up:
                uid = add(r['target_table'], scol, r.get('target_schema',''))
                ek  = f'{uid}>{pid}'
                if ek not in edges: edges[ek] = {'from':uid,'to':pid}
                nrows[uid].append(r)
                if uid not in visited:
                    visited.add(uid)
                    sub = _get(r['target_table'], scol)
                    nrows[uid].extend(sub)
                    for sr in sub:
                        q.append((sr.get('source_table','') or '',
                                  sr.get('source_column','') or '', uid, [sr]))
            continue

        if not stbl and not scol:
            continue

        sc_v   = _schema(stbl)
        src_id = add(stbl, scol, sc_v)
        ek     = f'{src_id}>{pid}'
        if ek not in edges: edges[ek] = {'from':src_id,'to':pid}
        if src_id in visited: continue
        visited.add(src_id)
        sub = _get(stbl, scol)
        nrows[src_id].extend(sub)
        for sr in sub:
            q.append((sr.get('source_table','') or '',
                      sr.get('source_column','') or '', src_id, [sr]))

    return {'nodes':list(nodes.values()),
            'edges':list(edges.values()),
            'nrows':nrows}

SAFE = ['target_table','target_schema','sub_target_table','sub_target_schema',
        'target_column','source_table','source_schema','source_column',
        'process','sql_process','logic']

def clean_nrows(nrows):
    """Deduplicate and sanitise node row dicts for JSON embedding."""
    out = {}
    for k, rv in nrows.items():
        seen_set, ded = set(), []
        for r in rv:
            rk = (r.get('sql_process',''), r.get('source_table',''),
                  r.get('source_column',''), r.get('target_column',''))
            if rk not in seen_set:
                seen_set.add(rk)
                ded.append({f: str(r.get(f,'') or '') for f in SAFE})
        out[k] = ded
    return out

def make_html(gd, title, subtitle, ts_str):
    """Substitute placeholders into the HTML template and return final HTML string."""
    gd_json = json.dumps(gd, ensure_ascii=False)
    return (HTML_TEMPLATE
            .replace('__TITLE__',    title)
            .replace('__SUBTITLE__', subtitle)
            .replace('__TS__',       ts_str)
            .replace('__GD__',       gd_json))

def safe_name(s): return re.sub(r'[^a-zA-Z0-9_-]', '_', str(s).strip())

print('Helpers ready')


## 6. HTML Template

In [ ]:
# HTML_TEMPLATE is built from HTML_LINES — inject it here so make_html() can use it
# (This cell re-defines it from the same list to avoid triple-quote nesting in helpers cell)
HTML_TEMPLATE = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<title>__TITLE__ — IKG Lineage</title>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/sigma.js/2.4.0/sigma.min.js"></script>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/graphology/0.25.4/graphology.umd.min.js"></script>\n<style>\n*{box-sizing:border-box;margin:0;padding:0;}\nbody{font-family:Inter,Segoe UI,Arial,sans-serif;height:100vh;display:flex;flex-direction:column;overflow:hidden;background:#f4f7fb;}\n#topbar{background:linear-gradient(120deg,#0d1f3c 0%,#1a3a6e 55%,#1565c0 100%);color:#fff;padding:10px 22px;display:flex;align-items:center;gap:14px;box-shadow:0 2px 14px rgba(0,0,0,.32);flex-shrink:0;z-index:20;}\n#topbar h1{font-size:16px;font-weight:800;letter-spacing:.3px;white-space:nowrap;}\n#topbar h1 span{color:#90caf9;}\n#topbar .sep{width:1px;height:26px;background:rgba(255,255,255,.2);flex-shrink:0;}\n#topbar .meta{font-size:12px;color:rgba(255,255,255,.62);}\n#topbar .meta b{color:rgba(255,255,255,.92);font-weight:700;}\n#ts{font-size:11px;color:rgba(255,255,255,.38);margin-left:auto;white-space:nowrap;}\n#main{display:flex;flex:1;overflow:hidden;}\n#gwrap{flex:1;position:relative;background:linear-gradient(150deg,#ecf1fb 0%,#e4ecf8 100%);overflow:hidden;}\n#sc{width:100%;height:100%;}\n#cam{position:absolute;bottom:18px;right:18px;display:flex;flex-direction:column;gap:6px;}\n.cbtn{width:34px;height:34px;background:#fff;border:1.5px solid #b8cde8;color:#1a3a6e;border-radius:8px;cursor:pointer;font-size:18px;font-weight:800;display:flex;align-items:center;justify-content:center;box-shadow:0 2px 8px rgba(0,0,0,.12);transition:all .15s;}\n.cbtn:hover{background:#1565c0;color:#fff;border-color:#1565c0;}\n#legend{position:absolute;bottom:18px;left:18px;background:rgba(255,255,255,.94);border:1px solid #d0daf0;border-radius:10px;padding:11px 15px;box-shadow:0 3px 14px rgba(0,0,0,.1);max-width:230px;backdrop-filter:blur(6px);}\n#legend h4{font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.6px;color:#4a6080;margin-bottom:8px;}\n.lr{display:flex;align-items:center;gap:9px;margin-bottom:5px;font-size:12px;color:#263248;}\n.ld{width:14px;height:14px;border-radius:4px;flex-shrink:0;border:2px solid rgba(0,0,0,.15);}\n#tip{position:absolute;background:#1a2744;color:#fff;padding:8px 13px;border-radius:8px;font-size:12px;display:none;pointer-events:none;white-space:nowrap;z-index:30;box-shadow:0 4px 16px rgba(0,0,0,.28);}\n#tip .tt{font-weight:700;font-size:13px;color:#90caf9;letter-spacing:.2px;}\n#tip .tc{color:rgba(255,255,255,.68);font-size:11px;margin-top:2px;}\n#loader{position:absolute;inset:0;background:rgba(244,247,251,.78);display:none;align-items:center;justify-content:center;z-index:50;flex-direction:column;gap:14px;}\n#loader.on{display:flex;}\n.spin{width:44px;height:44px;border:4px solid #c0cfe8;border-top-color:#1565c0;border-radius:50%;animation:rot .75s linear infinite;}\n@keyframes rot{to{transform:rotate(360deg);}}\n#loader p{color:#2e4a6e;font-size:14px;font-weight:500;}\n#rpanel{width:340px;background:#fff;border-left:1.5px solid #d4deee;display:flex;flex-direction:column;flex-shrink:0;overflow:hidden;}\n#rphdr{background:linear-gradient(90deg,#0d1f3c,#1a3a6e);color:#fff;padding:11px 16px;font-size:13px;font-weight:700;letter-spacing:.2px;flex-shrink:0;display:flex;align-items:center;gap:8px;}\n#rpbody{flex:1;overflow-y:auto;padding:14px;}\n.placeholder{display:flex;flex-direction:column;align-items:center;justify-content:center;height:100%;gap:10px;color:#7a92b0;text-align:center;padding:30px;}\n.placeholder .icon{font-size:44px;opacity:.45;}\n.placeholder p{font-size:13px;line-height:1.6;}\n.ncard{margin-bottom:10px;border:1.5px solid #d4deee;border-radius:10px;overflow:hidden;box-shadow:0 1px 6px rgba(0,0,0,.05);}\n.ncard-hdr{padding:10px 13px;display:flex;align-items:flex-start;gap:8px;flex-wrap:wrap;}\n.ncard-tbl{font-size:14px;font-weight:800;color:#fff;letter-spacing:.2px;display:block;}\n.ncard-col{font-size:12px;color:rgba(255,255,255,.72);font-weight:500;font-style:italic;display:block;}\n.ncard-body{}\n.nrow{display:flex;padding:5px 13px;border-bottom:1px solid #eef2f9;font-size:13px;}\n.nrow:last-child{border-bottom:none;}\n.nlbl{color:#607090;width:120px;flex-shrink:0;font-size:11.5px;font-weight:600;letter-spacing:.1px;}\n.nval{color:#1a2744;word-break:break-word;font-weight:500;}\n.nval.empty{color:#b0bec8;font-style:italic;font-weight:400;}\n.nval.mono{font-family:Courier New,monospace;font-size:11px;background:#f2f6fc;padding:2px 6px;border-radius:4px;color:#1a3a6e;}\n.rbadge{display:inline-flex;align-items:center;padding:2px 9px;border-radius:9px;font-size:11px;font-weight:700;letter-spacing:.3px;}\n.rb-sel{background:#dbeafe;color:#1d4ed8;}\n.rb-join{background:#d1fae5;color:#065f46;}\n.rb-wh{background:#fef3c7;color:#92400e;}\n.rb-hav{background:#fce7f3;color:#9d174d;}\n.rb-val{background:#ede9fe;color:#5b21b6;}\n.rb-str{background:#cffafe;color:#155e75;}\n#rpbody::-webkit-scrollbar{width:5px;}\n#rpbody::-webkit-scrollbar-thumb{background:#c0cfe8;border-radius:3px;}\n</style>\n</head>\n<body>\n<div id="topbar">\n  <h1>IKG Lineage <span>Explorer</span></h1>\n  <div class="sep"></div>\n  <div class="meta">__SUBTITLE__</div>\n  <div class="sep"></div>\n  <div class="meta">Nodes: <b id="nc">-</b>&nbsp;&nbsp;Edges: <b id="ec">-</b></div>\n  <div id="ts">__TS__</div>\n</div>\n<div id="main">\n  <div id="gwrap">\n    <div id="sc"></div>\n    <div id="loader"><div class="spin"></div><p>Building lineage graph...</p></div>\n    <div id="cam">\n      <button class="cbtn" title="Zoom in"  onclick="camZ(1.35)">+</button>\n      <button class="cbtn" title="Zoom out" onclick="camZ(0.74)">-</button>\n      <button class="cbtn" title="Fit"      onclick="camFit()">&#8861;</button>\n    </div>\n    <div id="legend"><h4>Schema</h4><div id="leg"></div></div>\n    <div id="tip"><div class="tt" id="tt-t"></div><div class="tc" id="tt-c"></div></div>\n  </div>\n  <div id="rpanel">\n    <div id="rphdr">&#128202; Node Details</div>\n    <div id="rpbody">\n      <div class="placeholder"><div class="icon">&#128269;</div><p>Click any node to see<br>lineage details here.</p></div>\n    </div>\n  </div>\n</div>\n<script>\nconst GD = __GD__;\nconst PALETTE=[\n  ["core_ikg",                       {bg:"#1565c0",br:"#0d47a1"}],\n  ["ikg_schema",                     {bg:"#1565c0",br:"#0d47a1"}],\n  ["sandbox_prj_smart_insights",     {bg:"#1565c0",br:"#0d47a1"}],\n  ["sandbox_ikg_pre_prd",            {bg:"#1976d2",br:"#1565c0"}],\n  ["core_wma_shared",                {bg:"#00796b",br:"#004d40"}],\n  ["edw_view_input_schema",          {bg:"#00796b",br:"#004d40"}],\n  ["ikg_vendor_schema",              {bg:"#00796b",br:"#004d40"}],\n  ["edw_input_schema",               {bg:"#0288d1",br:"#01579b"}],\n  ["core_wma_shared_masked",         {bg:"#00897b",br:"#00695c"}],\n  ["sandbox_wma_shared",             {bg:"#388e3c",br:"#1b5e20"}],\n  ["sandbox_prj_ds_data",            {bg:"#f57c00",br:"#e65100"}],\n  ["core_nlg",                       {bg:"#7b1fa2",br:"#4a148c"}],\n  ["nlg_schema",                     {bg:"#7b1fa2",br:"#4a148c"}],\n  ["core_model",                     {bg:"#5d4037",br:"#3e2723"}],\n  ["model_schema",                   {bg:"#5d4037",br:"#3e2723"}],\n  ["sandbox_prj_sbl",                {bg:"#6a1b9a",br:"#4a148c"}],\n  ["sandbox_prj_dsforoverdrive",     {bg:"#0277bd",br:"#01579b"}],\n  ["core_in_shared",                 {bg:"#6d4c41",br:"#4e342e"}],\n  ["ikg_clip_schema",                {bg:"#6d4c41",br:"#4e342e"}],\n  ["sandbox_prj_adhoc",              {bg:"#ad1457",br:"#880e4f"}],\n  ["sandbox_prj_smart_relationship", {bg:"#00838f",br:"#006064"}],\n  ["ikg_wealthx_schema",             {bg:"#00838f",br:"#006064"}],\n  ["sandbox_prj_rbat",               {bg:"#558b2f",br:"#33691e"}],\n];\nconst DEF = {bg:"#455a64",br:"#263238"};\nfunction theme(s){\n  const sl = (s||String()).toLowerCase().trim();\n  for(const [k,v] of PALETTE) if(sl===k||sl.includes(k)||k.includes(sl)) return v;\n  return DEF;\n}\nlet sig=null, G=null;\nwindow.onload = function(){ boot(); };\nfunction boot(){\n  document.getElementById("loader").classList.add("on");\n  setTimeout(function(){ try{ _build(); }catch(ex){ console.error(ex); } document.getElementById("loader").classList.remove("on"); }, 40);\n}\nfunction _build(){\n  if(sig){ sig.kill(); sig=null; }\n  G = new graphology.Graph({type:"directed", multi:false});\n  var nodes = GD.nodes, edges = GD.edges, nrows = GD.nrows;\n  // Kahn topological sort for level assignment\n  var aO = new Map(), id_ = new Map();\n  nodes.forEach(function(n){ aO.set(n.id,[]); id_.set(n.id,0); });\n  edges.forEach(function(e){\n    if(aO.has(e.from)&&aO.has(e.to)){\n      aO.get(e.from).push(e.to);\n      id_.set(e.to,(id_.get(e.to)||0)+1);\n    }\n  });\n  var lv = new Map(), q = [];\n  id_.forEach(function(d,id){ if(d===0) q.push(id); });\n  while(q.length){\n    var id=q.shift(), l=lv.get(id)||0;\n    (aO.get(id)||[]).forEach(function(t){\n      lv.set(t, Math.max(lv.get(t)||0, l+1));\n      id_.set(t, id_.get(t)-1);\n      if(id_.get(t)===0) q.push(t);\n    });\n  }\n  nodes.forEach(function(n){ if(!lv.has(n.id)) lv.set(n.id,0); });\n  var bL = new Map();\n  nodes.forEach(function(n){\n    var l=lv.get(n.id)||0;\n    if(!bL.has(l)) bL.set(l,[]);\n    bL.get(l).push(n);\n  });\n  var maxL = 0; bL.forEach(function(_,k){ if(k>maxL) maxL=k; });\n  var W=1100, H=680, PAD=110;\n  var xS = maxL>0 ? (W-PAD*2)/maxL : W/2;\n  var schemas = new Map();\n  bL.forEach(function(ns, l){\n    var x = PAD + (maxL-l)*xS;\n    ns.forEach(function(n, i){\n      var y = (i+1)*(H/(ns.length+1));\n      var th = theme(n.schema);\n      var sk = (n.schema||"unknown").toLowerCase();\n      if(!schemas.has(sk)) schemas.set(sk, {label: n.schema||"unknown", bg: th.bg});\n      var lbl = n.tbl + "|" + n.col;\n      G.addNode(n.id, {\n        x: x, y: y,\n        size: n.is_start ? 20 : 13,\n        color: th.bg,\n        label: lbl,\n        _tbl: n.tbl, _col: n.col, _schema: n.schema, _s: n.is_start\n      });\n    });\n  });\n  var eS = new Set();\n  edges.forEach(function(e, i){\n    var k = e.from + ">" + e.to;\n    if(G.hasNode(e.from) && G.hasNode(e.to) && !eS.has(k)){\n      eS.add(k);\n      G.addEdge(e.from, e.to, {size:2.5, color:"rgba(80,120,200,0.50)", type:"arrow"});\n    }\n  });\n  var container = document.getElementById("sc");\n  sig = new Sigma(G, container, {\n    renderEdgeLabels: false,\n    defaultEdgeType: "arrow",\n    labelFont: "Inter",\n    labelWeight: "700",\n    labelColor: {color: "#1a2744"},\n    labelSize: 11,\n    labelDensity: 1,\n    labelGridCellSize: 120,\n    minCameraRatio: 0.03,\n    maxCameraRatio: 15,\n    nodeReducer: function(node, data){\n      return Object.assign({}, data, {label: data._tbl + " / " + data._col});\n    }\n  });\n  // Custom label rendering - table name bold, column italic below\n  sig.on("afterRender", function(){});\n  sig.on("clickNode", function(e){ detail(e.node); });\n  var tip = document.getElementById("tip");\n  sig.on("enterNode", function(e){\n    var a = G.getNodeAttributes(e.node);\n    document.getElementById("tt-t").textContent = a._tbl;\n    document.getElementById("tt-c").textContent = "col: " + a._col;\n    tip.style.display = "block";\n    mv(e.event.original);\n  });\n  sig.on("leaveNode", function(){ tip.style.display = "none"; });\n  container.addEventListener("mousemove", function(ev){\n    if(tip.style.display !== "none") mv(ev);\n  });\n  function mv(ev){\n    var r = container.getBoundingClientRect();\n    tip.style.left = (ev.clientX - r.left + 15) + "px";\n    tip.style.top  = (ev.clientY - r.top  - 10) + "px";\n  }\n  var lb = document.getElementById("leg");\n  var legHtml = "";\n  var seen = [];\n  schemas.forEach(function(v, k){ seen.push(v); });\n  seen.slice(0,12).forEach(function(s){\n    legHtml += \'<div class="lr"><div class="ld" style="background:\' + s.bg + \'"></div><span>\' + esc(s.label) + \'</span></div>\';\n  });\n  lb.innerHTML = legHtml;\n  document.getElementById("nc").textContent = G.order;\n  document.getElementById("ec").textContent = G.size;\n  camFit();\n}\nfunction detail(nodeId){\n  var nrows = GD.nrows;\n  var rows = nrows[nodeId] || [];\n  var a = G.getNodeAttributes(nodeId);\n  var th = theme(a._schema);\n  var body = document.getElementById("rpbody");\n  // Deduplicate rows\n  var seen = new Set(), deduped = [];\n  rows.forEach(function(r){\n    var k = r.sql_process + "|" + r.source_table + "|" + r.source_column + "|" + r.target_column;\n    if(!seen.has(k)){ seen.add(k); deduped.push(r); }\n  });\n  // Node identity card\n  var h = \'<div class="ncard">\'\n    + \'<div class="ncard-hdr" style="background:\' + th.bg + \';border-bottom:3px solid \' + th.br + \'">\'\n    + \'<div><span class="ncard-tbl">&#128200; \' + esc(a._tbl) + \'</span>\'\n    + \'<span class="ncard-col">column: \' + esc(a._col) + \'</span></div>\'\n    + \'</div><div class="ncard-body">\'\n    + nr("Schema", a._schema)\n    + nr("Role", a._s ? "Profile Target (start)" : "Source / Intermediate")\n    + \'</div></div>\';\n  if(!deduped.length){\n    h += \'<div style="color:#7a92b0;font-size:13px;padding:18px;text-align:center;line-height:1.7">\'\n       + \'<div style="font-size:32px;margin-bottom:8px">&#128204;</div>\'\n       + \'Base source - no further upstream lineage.</div>\';\n    body.innerHTML = h; return;\n  }\n  deduped.forEach(function(r, i){\n    var proc = r.sql_process || "select";\n    h += \'<div class="ncard">\'\n      + \'<div class="ncard-hdr" style="background:\' + th.bg + \'22;border-bottom:2px solid \' + th.br + \'33">\'\n      + bdg(proc)\n      + \'<span style="font-size:11px;color:#4a6080;font-weight:600;margin-left:4px">Record \' + (i+1) + \' / \' + deduped.length + \'</span>\'\n      + \'</div><div class="ncard-body">\'\n      + nr("Target Table",      r.target_table)\n      + nr("Target Schema",     r.target_schema)\n      + nr("Sub-Target Table",  r.sub_target_table)\n      + nr("Sub-Target Schema", r.sub_target_schema)\n      + nr("Target Column",     r.target_column)\n      + nr("Source Table",      r.source_table)\n      + nr("Source Schema",     r.source_schema)\n      + nr("Source Column",     r.source_column)\n      + nr("Process",           r.process)\n      + nr("SQL Process",       r.sql_process)\n      + (r.logic ? nrm("Logic", r.logic.slice(0,450)) : "")\n      + \'</div></div>\';\n  });\n  body.innerHTML = h;\n}\nfunction nr(l,v){\n  var empty = !v || !String(v).trim();\n  return \'<div class="nrow"><span class="nlbl">\' + l + \'</span><span class="nval\' + (empty ? \' empty\' : \'\') + \'">\' + (empty ? \'-\' : esc(String(v))) + \'</span></div>\';\n}\nfunction nrm(l,v){\n  return \'<div class="nrow"><span class="nlbl">\' + l + \'</span><span class="nval mono">\' + esc(String(v)) + \'</span></div>\';\n}\nfunction bdg(p){\n  var m = {"select":"rb-sel","select-value":"rb-val","select*":"rb-str","join":"rb-join","where":"rb-wh","having":"rb-hav","where-subquery":"rb-wh"};\n  return \'<span class="rbadge \' + (m[p]||"rb-sel") + \'">\' + esc(p||"select") + \'</span>\';\n}\nfunction camZ(f){ if(sig) sig.getCamera().animatedZoom({duration:200, factor:f}); }\nfunction camFit(){ if(sig) sig.getCamera().animatedReset({duration:450}); }\nfunction esc(s){ return String(s||String()).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;"); }\n</script>\n</body>\n</html>'
print(f'HTML template ready: {len(HTML_TEMPLATE):,} chars')


## Mode 1 — Column / Table Lineage

1. **Type** to search a column in the dropdown (auto-filters blanks/nulls)
2. Select a **profile table** from the second dropdown
3. Click **▶ Trace** → saves `<table>_<column>_lineage.html` next to the notebook


In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────
_s1 = widgets.Output()

w_col = widgets.Combobox(
    options       = COLS_LIST,
    value         = '',
    placeholder   = 'Type to search column...',
    description   = 'Column:',
    ensure_option = True,
    style         = {'description_width': '80px'},
    layout        = widgets.Layout(width='420px'),
)

w_tbl = widgets.Combobox(
    options       = [],
    value         = '',
    placeholder   = 'Select a column first...',
    description   = 'Table:',
    ensure_option = True,
    style         = {'description_width': '80px'},
    layout        = widgets.Layout(width='420px'),
    disabled      = True,
)

btn1 = widgets.Button(
    description  = '\u25b6  Trace',
    button_style = 'primary',
    layout       = widgets.Layout(width='130px', height='36px'),
    disabled     = True,
)

def _on_col(change):
    col = (change['new'] or '').strip()
    with _s1:
        clear_output()
    w_tbl.value    = ''
    btn1.disabled  = True
    if not col:
        w_tbl.options  = []
        w_tbl.disabled = True
        return
    tbls = sorted(df_profile[df_profile['target_column'] == col]['target_table']
                  .dropna().unique().tolist())
    w_tbl.options  = tbls
    w_tbl.disabled = False
    w_tbl.placeholder = f'{len(tbls)} table(s) — type to search...'
    with _s1:
        print(f'Found {len(tbls)} profile table(s) for column: {col}')

def _on_tbl(change):
    btn1.disabled = not bool((change['new'] or '').strip())

w_col.observe(_on_col, names='value')
w_tbl.observe(_on_tbl, names='value')

def _trace1(_):
    col = (w_col.value or '').strip()
    tbl = (w_tbl.value or '').strip()
    if not col or not tbl:
        return
    with _s1:
        clear_output()
        print(f'Tracing lineage for  {tbl}.{col} ...')
    res = trace(col, tbl)
    if not res:
        with _s1:
            clear_output()
            print(f'No lineage found for {tbl}.{col}')
        return
    nrows_c = clean_nrows(res['nrows'])
    gd = {'nodes': res['nodes'], 'edges': res['edges'], 'nrows': nrows_c}
    ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    sub = f'Table: <b>{tbl}</b>&nbsp;&nbsp;Column: <b>{col}</b>'
    out = make_html(gd, f'{tbl}.{col}', sub, ts)
    fname = safe_name(tbl) + '_' + safe_name(col) + '_lineage.html'
    fpath = os.path.abspath(fname)
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(out)
    with _s1:
        clear_output()
        print(f'{len(res["nodes"])} nodes, {len(res["edges"])} edges')
        print(f'Saved: {fpath}')
        print('Open in Chrome / Edge / Firefox.')

btn1.on_click(_trace1)
ipy_display(widgets.VBox([widgets.HBox([w_col, w_tbl, btn1]), _s1]))


## Mode 2 — Insight Type Lineage

1. **Type** to search an ODM insight type in the dropdown
2. Click **▶ Trace** → queries `core_ikg.odm_rule_metadata_auto_refresh` for all
   `(profile_table, rule_column)` pairs, traces each, and saves
   `<insight_type>_lineage.html` next to the notebook


In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────
_s2 = widgets.Output()

w_ins = widgets.Combobox(
    options       = IT_LIST,
    value         = '',
    placeholder   = 'Type to search insight type...',
    description   = 'Insight Type:',
    ensure_option = True,
    style         = {'description_width': '100px'},
    layout        = widgets.Layout(width='480px'),
)

btn2 = widgets.Button(
    description  = '\u25b6  Trace',
    button_style = 'primary',
    layout       = widgets.Layout(width='130px', height='36px'),
    disabled     = True,
)

def _on_ins(change):
    btn2.disabled = not bool((change['new'] or '').strip())
    with _s2:
        clear_output()
        if (change['new'] or '').strip():
            print(f'Selected: {change["new"]}  -- click Trace')

w_ins.observe(_on_ins, names='value')

def _trace2(_):
    sel = (w_ins.value or '').strip()
    if not sel: return
    with _s2:
        clear_output()
        print(f'Loading ODM metadata for: {sel} ...')

    # Query (profile_table, rule_column) pairs
    q = ('SELECT DISTINCT profile_table, rule_column FROM '
         + ODM_SCHEMA + '.' + ODM_TABLE
         + " WHERE insight_type = %(s)s"
         + " AND profile_table IS NOT NULL AND profile_table <> ''"
         + " AND rule_column   IS NOT NULL AND rule_column   <> ''")
    df_scope = pd.read_sql(q, engine, params={'s': sel})

    if df_scope.empty:
        with _s2:
            clear_output()
            print(f'No metadata found for: {sel}')
        return

    with _s2:
        clear_output()
        print(f'{len(df_scope)} column-table pairs -- tracing lineage...')

    # Trace lineage for every pair and merge into one graph
    all_nodes, all_edges, all_nrows = {}, {}, {}
    for _, row in df_scope.iterrows():
        ptbl = str(row['profile_table']).strip()
        rcol = str(row['rule_column']).strip()
        res  = trace(rcol, ptbl)
        if not res: continue
        for n in res['nodes']:
            if n['id'] not in all_nodes: all_nodes[n['id']] = n
            elif n['is_start']:          all_nodes[n['id']]['is_start'] = True
        for e in res['edges']:
            ek = f'{e["from"]}>{e["to"]}'
            if ek not in all_edges: all_edges[ek] = e
        for k, rv in res['nrows'].items():
            all_nrows.setdefault(k, [])
            all_nrows[k].extend(rv)

    nodes_list = list(all_nodes.values())
    edges_list = list(all_edges.values())
    if not nodes_list:
        with _s2:
            clear_output()
            print('No lineage nodes found.')
        return

    nrows_c = clean_nrows(all_nrows)
    gd  = {'nodes': nodes_list, 'edges': edges_list, 'nrows': nrows_c}
    ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    sub = f'Insight: <b>{sel}</b>&nbsp;&nbsp;Pairs: <b>{len(df_scope)}</b>'
    out = make_html(gd, sel, sub, ts)
    fname = safe_name(sel) + '_lineage.html'
    fpath = os.path.abspath(fname)
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(out)
    with _s2:
        clear_output()
        print(f'{len(nodes_list)} nodes, {len(edges_list)} edges')
        print(f'Saved: {fpath}')
        print('Open in Chrome / Edge / Firefox.')

btn2.on_click(_trace2)
ipy_display(widgets.VBox([widgets.HBox([w_ins, btn2]), _s2]))


## Usage Guide

### Mode 1 — Column / Table

| Step | Action |
|------|--------|
| 1 | Run cells 1–5 |
| 2 | Run Mode 1 cell |
| 3 | Type to search column (blank/null values auto-excluded) |
| 4 | Select profile table from second dropdown |
| 5 | Click **Trace** — opens `<table>_<column>_lineage.html` |

### Mode 2 — Insight Type

| Step | Action |
|------|--------|
| 1 | Run Mode 2 cell |
| 2 | Type to search insight type |
| 3 | Click **Trace** — opens `<insight_type>_lineage.html` |

### HTML Graph

- **Click any node** → right panel shows all 11 lineage fields
- **+ / − / ⊡** → zoom in / out / fit
- **Node colour** = schema (see legend in graph bottom-left)
- **Larger node** = profile target (start point)
- **Direction** = edges flow left (source) → right (target)

### Schema colours (no red)

| Schema | Colour |
|--------|--------|
| `core_ikg` / `sandbox_prj_smart_insights` | Deep Blue |
| `core_wma_shared` / `edw_view_input_schema` | Teal |
| `edw_input_schema` | Sky Blue |
| `core_nlg` | Purple |
| `core_model` | Brown |
| `sandbox_prj_ds_data` | Orange |
| `sandbox_wma_shared` | Green |
| `sandbox_prj_smart_relationship` | Cyan |
